### Cell 1: Configuration & Widgets

In [0]:
# Widgets
dbutils.widgets.text("input_folder", "/Volumes/1_inland/sectra/vone/Hackathon3/", "Input DICOM Folder")
dbutils.widgets.text("ocr_log_table", "8_dev.pacs.image_ocr_text", "OCR Log Table")
dbutils.widgets.text("preprocessing", "True", "OCR Preprocessing (True/False)")
dbutils.widgets.text("min_text_length", "3", "Min OCR Text Length")
dbutils.widgets.text("padding", "10", "Redaction Padding (px)")
dbutils.widgets.text("batch_size", "50", "Images per worker batch")

# Environment variables — must be set before PaddleOCR import
import os
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.8"
os.environ.setdefault("PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT", "False")
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

In [0]:
%pip install SimpleITK

In [0]:
# Restart Python so env vars take effect before paddle imports
dbutils.library.restartPython()

### Cell 2: Imports

In [0]:
import os
import re
import math
import calendar
import datetime
import json
from collections import defaultdict
from pathlib import Path
from typing import Iterator

import numpy as np
import cv2
import SimpleITK as sitk
import pandas as pd

from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType, TimestampType
)

# Read widget values
INPUT_FOLDER = dbutils.widgets.get("input_folder")
OCR_LOG_TABLE = dbutils.widgets.get("ocr_log_table")
PREPROCESSING = dbutils.widgets.get("preprocessing").strip().lower() == "true"
MIN_TEXT_LENGTH = int(dbutils.widgets.get("min_text_length"))
PADDING = int(dbutils.widgets.get("padding"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

# Derive output folder with timestamp
_run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FOLDER = f"/Volumes/8_dev/pacs/anon_images/{_run_ts}/"

print(f"Input folder:  {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"OCR log table: {OCR_LOG_TABLE}")
print(f"Preprocessing: {PREPROCESSING}")
print(f"Min text len:  {MIN_TEXT_LENGTH}")
print(f"Padding:       {PADDING}")
print(f"Batch size:    {BATCH_SIZE}")

# Spark config for GPU scheduling and image workloads
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "50")

### Cell 3: Helper functions (reused from original)

In [0]:
# ── Image Helpers ─────────────────────────────────────────────────────────────

def _normalize_to_uint8(image_array):
    """Normalize any numeric array to uint8 0-255 for OCR input."""
    if image_array.ndim == 3:
        image_array = np.squeeze(image_array)
    if image_array.ndim == 3:
        image_array = image_array[0]
    img = image_array.astype(np.float32)
    img_min, img_max = img.min(), img.max()
    if img_max - img_min > 0:
        img = (img - img_min) / (img_max - img_min) * 255
    return img.astype(np.uint8)


def _preprocess(img_uint8):
    """Invert + CLAHE for white-on-dark ultrasound text."""
    inverted = cv2.bitwise_not(img_uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(inverted)


def _polygon_to_bbox(polygon):
    """Convert PaddleOCR quadrilateral to {x, y, width, height}."""
    xs = [pt[0] for pt in polygon]
    ys = [pt[1] for pt in polygon]
    x_min, x_max = float(min(xs)), float(max(xs))
    y_min, y_max = float(min(ys)), float(max(ys))
    return {"x": x_min, "y": y_min, "width": x_max - x_min, "height": y_max - y_min}


# ── OCR functions (used inside workers) ───────────────────────────────────────
# Compatible with PaddleOCR 2.8.x API: ocr.ocr(img) returns
#   [ [  [box, (text, conf)], [box, (text, conf)], ...  ] ]
# where box is [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
# Detection-only (rec=False) returns:
#   [ [ box, box, ... ] ]

def run_ocr_structured(ocr_engine, numpy_array, preprocessing=True):
    """Full OCR: detection + recognition. Returns dict with text and bounding boxes."""
    img = _normalize_to_uint8(numpy_array)
    if preprocessing:
        img = _preprocess(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    result = ocr_engine.ocr(img, cls=False)

    ocr_results = {}
    line_index = 0
    # result is a list of pages; typically one page for a single image
    if result and result[0]:
        for detection in result[0]:
            polygon = detection[0]        # [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
            text = detection[1][0]         # recognized text
            ocr_results[f"line_{line_index}"] = {
                "text": text,
                "bounding_box": _polygon_to_bbox(polygon),
                "has_pii": False,
                "pii_details": [],
            }
            line_index += 1
    return ocr_results


def run_detection_only(ocr_engine, numpy_array, preprocessing=True):
    """Detection only: returns bounding boxes for all text regions (blanket mode)."""
    img = _normalize_to_uint8(numpy_array)
    if preprocessing:
        img = _preprocess(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    result = ocr_engine.ocr(img, rec=False, cls=False)

    ocr_results = {}
    line_index = 0
    if result and result[0]:
        for polygon in result[0]:
            ocr_results[f"line_{line_index}"] = {
                "text": "",
                "bounding_box": _polygon_to_bbox(polygon),
                "has_pii": True,
                "pii_details": [],
            }
            line_index += 1
    return ocr_results


# ── Text Filter ───────────────────────────────────────────────────────────────

def filter_ocr_results(ocr_results, min_length=2, min_alpha_ratio=0.0):
    """Filter OCR results, marking short/non-text detections as non-PII."""
    filtered = {}
    for line_id, data in ocr_results.items():
        text = data["text"].strip()
        entry = {
            "text": data["text"],
            "bounding_box": data["bounding_box"],
            "pii_details": data.get("pii_details", []),
            "filtered": False,
            "filter_reason": None,
        }

        if len(text) < min_length:
            entry["has_pii"] = False
            entry["filtered"] = True
            entry["filter_reason"] = f"too_short (len={len(text)} < {min_length})"
            filtered[line_id] = entry
            continue

        if min_alpha_ratio > 0:
            total = len(text.replace(" ", ""))
            alpha = sum(1 for c in text if c.isalpha())
            ratio = alpha / max(total, 1)
            if ratio < min_alpha_ratio:
                entry["has_pii"] = False
                entry["filtered"] = True
                entry["filter_reason"] = f"low_alpha (ratio={ratio:.0%} < {min_alpha_ratio:.0%})"
                filtered[line_id] = entry
                continue

        entry["has_pii"] = True
        filtered[line_id] = entry

    return filtered


# ── PII Detection ─────────────────────────────────────────────────────────────

_PII_PATTERNS = [
    ("MRN", re.compile(r"\bMRN[:\s]*\d{6,10}\b", re.IGNORECASE)),
    ("ACCESSION", re.compile(r"\bACC(?:ESSION)?[:\s#]*\w{4,15}\b", re.IGNORECASE)),
    ("PATIENT_ID", re.compile(r"\bPID[:\s]*\d{4,12}\b", re.IGNORECASE)),
    ("GENDER", re.compile(r"\b(?:Male|Female|MALE|FEMALE)\b")),
    ("DATE", re.compile(
        r"\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b"
        r"|\b\d{4}[/\-\.]\d{1,2}[/\-\.]\d{1,2}\b"
        r"|\b\d{8}\b"
    )),
]

_ULTRASOUND_ALLOWLIST = {
    "sos", "map", "b/0", "b/1", "b/2", "m/0", "m/1",
    "logiq", "voluson", "vivid",
    "epiq", "affiniti", "iu22", "cx50",
    "acuson", "sequoia", "juniper", "helx",
    "aplio", "xario", "toshiba", "canon",
    "samsung", "rs85", "hs70", "hs60",
    "mindray", "resona", "dc-80", "dc-70",
    "arietta", "lisendo", "hitachi", "fujifilm",
    "frq", "freq", "gain", "tis", "tib", "tic", "tls",
    "prf", "wf", "thi", "cpa", "cpd", "pwr", "fps",
    "cw", "pw", "sv", "cf", "cci",
    "breast", "abdomen", "thyroid", "liver", "kidney", "ob",
    "cardiac", "vascular", "msk", "carotid", "renal",
    "dist", "area", "vol", "circ", "vel", "ari", "eri",
}


def is_valid_nhs_number(nhs_number):
    """Validate NHS number using checksum algorithm."""
    if not isinstance(nhs_number, str):
        return False
    nhs_digits = re.sub(r'[\s-]', '', nhs_number)
    if not nhs_digits.isdigit() or len(nhs_digits) != 10:
        return False
    weights = [10, 9, 8, 7, 6, 5, 4, 3, 2]
    total = sum(int(digit) * weight for digit, weight in zip(nhs_digits[:9], weights))
    remainder = total % 11
    check_digit = 11 - remainder
    if check_digit == 11:
        check_digit = 0
    elif check_digit == 10:
        return False
    return check_digit == int(nhs_digits[9])


def _build_dob_patterns(dob_dt):
    """Build regex patterns for common DOB renderings."""
    if dob_dt is None:
        return []

    day = dob_dt.day
    month = dob_dt.month
    year = dob_dt.year

    d = str(day)
    dd = f"{day:02d}"
    m = str(month)
    mm = f"{month:02d}"
    yyyy = str(year)
    yy = f"{year % 100:02d}"

    month_full = calendar.month_name[month]
    month_abbr = calendar.month_abbr[month]
    months_regex = f"(?:{re.escape(month_full)}|{re.escape(month_abbr)})"
    sep = r"[.\-/\s]"
    ord_suffix = r"(?:st|nd|rd|th)?"

    patterns = [
        fr"(?<!\d){dd}{sep}{mm}{sep}{yyyy}(?!\d)",
        fr"(?<!\d){d}{sep}{m}{sep}{yyyy}(?!\d)",
        fr"(?<!\d){dd}{sep}{mm}{sep}{yy}(?!\d)",
        fr"(?<!\d){d}{sep}{m}{sep}{yy}(?!\d)",
        fr"(?<!\d){yyyy}{sep}{mm}{sep}{dd}(?!\d)",
        fr"(?<!\d){yyyy}{sep}{m}{sep}{d}(?!\d)",
        fr"(?<!\d){dd}{mm}{yyyy}(?!\d)",
        fr"(?<!\d){yyyy}{mm}{dd}(?!\d)",
        fr"(?<!\d){dd}{mm}{yy}(?!\d)",
        fr"\b{d}{ord_suffix}{sep}+{months_regex}{sep}+{yyyy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yyyy}\b",
        fr"\b{months_regex}{sep}+{d}{ord_suffix}{sep}+{yyyy}\b",
        fr"\b{months_regex}{sep}+{dd}{ord_suffix}{sep}+{yyyy}\b",
        fr"\b{d}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
        fr"\b{months_regex}{sep}+{d}{ord_suffix}{sep}+{yy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
    ]
    return patterns


def detect_pii_in_ocr(ocr_results, deny_list):
    """Classify OCR text lines as PII using deny-list matching + regex patterns."""
    deny_set = {s.strip().lower() for s in deny_list if s and s.strip()}

    pii_results = {}
    for line_id, data in ocr_results.items():
        text = data["text"]
        found_entities = []

        # 1. Deny-list matching (case-insensitive substring)
        text_lower = text.strip().lower()
        for deny_term in deny_set:
            if len(deny_term) < 2:
                continue
            if deny_term in text_lower:
                found_entities.append({
                    "entity_text": text,
                    "type": "PHI_MATCH",
                    "source": "deny_list",
                    "matched_term": deny_term,
                })
                break

        # 2. Regex patterns
        for pattern_name, pattern in _PII_PATTERNS:
            for match in pattern.finditer(text):
                matched = match.group()
                if matched.strip().lower() in _ULTRASOUND_ALLOWLIST:
                    continue
                found_entities.append({
                    "entity_text": matched,
                    "type": pattern_name,
                    "source": "regex",
                })

        # 3. NHS number validation
        for m in re.finditer(r'(?<!\d)(\d[\s-]?){9}\d(?!\d)', text):
            raw = m.group()
            if is_valid_nhs_number(raw):
                found_entities.append({
                    "entity_text": raw,
                    "type": "NHS_NUMBER",
                    "source": "nhs_checksum",
                })

        pii_results[line_id] = {
            "text": text,
            "bounding_box": data["bounding_box"],
            "has_pii": len(found_entities) > 0,
            "pii_details": found_entities,
            "filtered": data.get("filtered", False),
            "filter_reason": data.get("filter_reason"),
        }

    return pii_results


# ── Redaction ─────────────────────────────────────────────────────────────────

def redact_pii_from_image(image_array, pii_report, padding=10):
    """Black-fill bounding boxes flagged as PII in the image array."""
    redacted = image_array.copy()
    img_h, img_w = redacted.shape[:2]
    fill_value = int(redacted.min())

    for item in pii_report.values():
        if not item.get("has_pii"):
            continue
        box = item["bounding_box"]
        x_start = int(max(0, math.floor(box["x"] - padding)))
        y_start = int(max(0, math.floor(box["y"] - padding)))
        x_end = int(min(img_w, math.ceil(box["x"] + box["width"] + padding)))
        y_end = int(min(img_h, math.ceil(box["y"] + box["height"] + padding)))
        redacted[y_start:y_end, x_start:x_end] = fill_value

    return redacted


# ── DICOM Metadata ────────────────────────────────────────────────────────────

_PII_DICOM_TAGS = [
    "0010|0010",  # PatientName
    "0010|0020",  # PatientID
    "0010|0030",  # PatientBirthDate
    "0010|0040",  # PatientSex
    "0010|1000",  # OtherPatientIDs
    "0010|1001",  # OtherPatientNames
    "0010|2160",  # EthnicGroup
    "0008|0050",  # AccessionNumber
    "0008|0080",  # InstitutionName
    "0008|0090",  # ReferringPhysicianName
    "0008|1050",  # PerformingPhysicianName
]

_DENY_LIST_DICOM_TAGS = {
    "0010|0010": "PatientName",
    "0010|0020": "PatientID",
    "0010|0030": "PatientBirthDate",
    "0008|0080": "InstitutionName",
    "0008|0090": "ReferringPhysicianName",
    "0008|0050": "AccessionNumber",
}


def _extract_metadata_deny_list(sitk_image):
    """Extract PII values from DICOM headers for use as deny-list terms."""
    deny_list = []
    for tag in _DENY_LIST_DICOM_TAGS:
        if sitk_image.HasMetaDataKey(tag):
            value = sitk_image.GetMetaData(tag).strip()
            if value:
                deny_list.append(value)
                if "^" in value:
                    deny_list.append(value.replace("^", " "))
                    for part in value.split("^"):
                        part = part.strip()
                        if part and len(part) > 1:
                            deny_list.append(part)
    return deny_list


def _extract_dicom_accession(sitk_image):
    tag = "0008|0050"
    if sitk_image.HasMetaDataKey(tag):
        return sitk_image.GetMetaData(tag).strip()
    return None


def _extract_dicom_study_uid(sitk_image):
    tag = "0020|000d"
    if sitk_image.HasMetaDataKey(tag):
        return sitk_image.GetMetaData(tag).strip()
    return None


def _extract_dicom_patient_id(sitk_image):
    tag = "0010|0020"
    if sitk_image.HasMetaDataKey(tag):
        return sitk_image.GetMetaData(tag).strip()
    return None


def strip_dicom_pii(sitk_image):
    """Remove PII tags from a SimpleITK image's DICOM metadata (in-place)."""
    for tag in _PII_DICOM_TAGS:
        if sitk_image.HasMetaDataKey(tag):
            sitk_image.SetMetaData(tag, "")
    return sitk_image

### Cell 4: Patient Linking & PHI (driver-side, runs once)

In [0]:
def _find_dicoms(root_dir):
    """Recursively find DICOM files in a volume directory."""
    exts = {".dcm", ".mhd", ".nii", ".nrrd"}
    for dirpath, _, filenames in os.walk(root_dir):
        for f in sorted(filenames):
            if Path(f).suffix.lower() in exts or "." not in f:
                yield os.path.join(dirpath, f)


def link_images_to_patients(file_headers):
    """Link DICOM files to patients via imaging_metadata.

    Multiple files can share the same accession/study_uid (e.g. multi-frame studies).
    Uses defaultdict(list) to map each key to ALL associated files.
    """
    accessions = defaultdict(list)
    study_uids = defaultdict(list)
    for fp, hdr in file_headers.items():
        if hdr.get("accession"):
            accessions[hdr["accession"]].append(fp)
        if hdr.get("study_uid"):
            study_uids[hdr["study_uid"]].append(fp)

    result = {fp: None for fp in file_headers}

    if not accessions and not study_uids:
        return result

    imaging_meta = spark.table("4_prod.pacs.imaging_metadata")

    # Try accession number first
    if accessions:
        acc_list = list(accessions.keys())
        matches = (
            imaging_meta
            .filter(col("AccessionNbr").isin(acc_list))
            .filter(col("PersonId").isNotNull())
            .select("AccessionNbr", "PersonId")
            .distinct()
            .collect()
        )
        for row in matches:
            for fp in accessions.get(row.AccessionNbr, []):
                if result[fp] is None:
                    result[fp] = row.PersonId

    # Fall back to StudyInstanceUID for still-unlinked files
    unlinked_uids = defaultdict(list)
    for uid, fps in study_uids.items():
        for fp in fps:
            if result.get(fp) is None:
                unlinked_uids[uid].append(fp)

    if unlinked_uids:
        uid_list = list(unlinked_uids.keys())
        matches = (
            imaging_meta
            .filter(col("ExaminationStudyUid").isin(uid_list))
            .filter(col("PersonId").isNotNull())
            .select("ExaminationStudyUid", "PersonId")
            .distinct()
            .collect()
        )
        for row in matches:
            for fp in unlinked_uids.get(row.ExaminationStudyUid, []):
                if result[fp] is None:
                    result[fp] = row.PersonId

    return result


def gather_patient_phi(person_ids):
    """Gather PHI for a set of person IDs from Millennium tables."""
    if not person_ids:
        return {}

    from pyspark.sql.functions import collect_set, when, collect_list, struct

    pid_list = list(set(person_ids))
    phi = {pid: {"first_names": [], "middle_names": [], "last_names": [],
                  "dob": None, "addresses": [], "aliases": []} for pid in pid_list}

    # Names
    names_df = (
        spark.table("4_prod.raw.mill_person_name")
        .filter(col("PERSON_ID").isin(pid_list))
        .groupBy("PERSON_ID")
        .agg(
            collect_set(when(col("NAME_FIRST").isNotNull(), col("NAME_FIRST"))).alias("first_names"),
            collect_set(when(col("NAME_MIDDLE").isNotNull(), col("NAME_MIDDLE"))).alias("middle_names"),
            collect_set(when(col("NAME_LAST").isNotNull(), col("NAME_LAST"))).alias("last_names"),
        )
        .collect()
    )
    for row in names_df:
        pid = row.PERSON_ID
        if pid in phi:
            phi[pid]["first_names"] = [n for n in (row.first_names or []) if n and n.strip()]
            phi[pid]["middle_names"] = [n for n in (row.middle_names or []) if n and n.strip()]
            phi[pid]["last_names"] = [n for n in (row.last_names or []) if n and n.strip()]

    # DOB
    dob_df = (
        spark.table("4_prod.raw.mill_person")
        .filter(col("PERSON_ID").isin(pid_list))
        .select("PERSON_ID", "BIRTH_DT_TM")
        .collect()
    )
    for row in dob_df:
        if row.PERSON_ID in phi and row.BIRTH_DT_TM:
            phi[row.PERSON_ID]["dob"] = row.BIRTH_DT_TM

    # Addresses
    addr_df = (
        spark.table("4_prod.raw.mill_address")
        .filter(col("PARENT_ENTITY_NAME") == "PERSON")
        .filter(col("PARENT_ENTITY_ID").isin(pid_list))
        .filter(col("ACTIVE_IND") == 1)
        .select(
            col("PARENT_ENTITY_ID").alias("PERSON_ID"),
            struct(
                "STREET_ADDR", "STREET_ADDR2", "STREET_ADDR3", "STREET_ADDR4",
                "CITY", "COUNTY", "STATE", "COUNTRY", "ZIPCODE", "POSTAL_IDENTIFIER"
            ).alias("address"),
        )
        .groupBy("PERSON_ID")
        .agg(collect_list("address").alias("addresses"))
        .collect()
    )
    for row in addr_df:
        if row.PERSON_ID in phi:
            phi[row.PERSON_ID]["addresses"] = row.addresses or []

    # Aliases (MRN, NHS, etc.)
    alias_df = (
        spark.table("4_prod.raw.mill_person_alias")
        .filter(col("PERSON_ID").isin(pid_list))
        .filter(col("ACTIVE_IND") == 1)
        .filter(col("ALIAS").isNotNull())
        .filter(col("ALIAS") != "")
        .groupBy("PERSON_ID")
        .agg(collect_set("ALIAS").alias("aliases"))
        .collect()
    )
    for row in alias_df:
        if row.PERSON_ID in phi:
            phi[row.PERSON_ID]["aliases"] = [a for a in (row.aliases or []) if a and a.strip()]

    return phi


def build_deny_list_from_phi(phi_dict, dicom_header_deny_list):
    """Build a combined deny-list from Millennium PHI + pre-extracted DICOM header values."""
    deny_list = list(dicom_header_deny_list)

    for name_list in [phi_dict.get("first_names", []),
                      phi_dict.get("middle_names", []),
                      phi_dict.get("last_names", [])]:
        for name in name_list:
            if name and len(name) > 1:
                deny_list.append(name)

    for alias in phi_dict.get("aliases", []):
        if alias and len(alias) > 2:
            deny_list.append(alias)

    dob = phi_dict.get("dob")
    if dob:
        dd = f"{dob.day:02d}"
        mm = f"{dob.month:02d}"
        yyyy = str(dob.year)
        deny_list.append(f"{dd}/{mm}/{yyyy}")
        deny_list.append(f"{dd}-{mm}-{yyyy}")
        deny_list.append(f"{dd}.{mm}.{yyyy}")
        deny_list.append(f"{yyyy}{mm}{dd}")
        deny_list.append(f"{dd}{mm}{yyyy}")

    for addr in phi_dict.get("addresses", []):
        for field in ["STREET_ADDR", "STREET_ADDR2", "STREET_ADDR3", "STREET_ADDR4",
                      "CITY", "COUNTY", "STATE", "COUNTRY", "ZIPCODE", "POSTAL_IDENTIFIER"]:
            val = addr[field] if isinstance(addr, dict) else getattr(addr, field, None)
            if val and len(str(val).strip()) > 2:
                deny_list.append(str(val).strip())

    return deny_list

In [0]:
# ── Phase 1: Discover files, read headers, link patients, gather PHI ─────────

print(f"Scanning {INPUT_FOLDER} for DICOM files...")
dicom_files = list(_find_dicoms(INPUT_FOLDER))
print(f"Found {len(dicom_files)} DICOM files")

# Read DICOM headers sequentially (lightweight, metadata only)
print("Reading DICOM headers for patient linking...")
file_headers = {}       # {path: {"accession", "study_uid", "patient_id"}}
dicom_deny_lists = {}   # {path: [deny_list_terms from DICOM header]}
valid_files = []
skipped = 0

for dcm_path in dicom_files:
    try:
        img = sitk.ReadImage(dcm_path)
        try:
            sitk.GetArrayFromImage(img)
        except Exception:
            skipped += 1
            continue
        file_headers[dcm_path] = {
            "accession": _extract_dicom_accession(img),
            "study_uid": _extract_dicom_study_uid(img),
            "patient_id": _extract_dicom_patient_id(img),
        }
        dicom_deny_lists[dcm_path] = _extract_metadata_deny_list(img)
        valid_files.append(dcm_path)
    except Exception as e:
        print(f"  Could not read {dcm_path}: {e}")
        skipped += 1

print(f"  {len(valid_files)} readable, {skipped} skipped (no pixel data or unreadable)")

# Link to patients
print("Linking images to patients via imaging_metadata...")
linking_map = link_images_to_patients(file_headers)
linked_count = sum(1 for v in linking_map.values() if v is not None)
unlinked_count = len(valid_files) - linked_count
print(f"  {linked_count} linked, {unlinked_count} unlinked (will blanket-redact)")

# Gather PHI for linked patients
linked_person_ids = [pid for pid in linking_map.values() if pid is not None]
phi_data = {}
if linked_person_ids:
    print("Gathering PHI for linked patients...")
    phi_data = gather_patient_phi(linked_person_ids)
    print(f"  PHI gathered for {len(phi_data)} patients")

# Build per-file deny lists (merge DICOM header + patient PHI)
file_deny_lists = {}   # {path: list[str]}
file_dob_patterns = {} # {path: list[str]}
for fp in valid_files:
    person_id = linking_map.get(fp)
    header_deny = dicom_deny_lists.get(fp, [])
    if person_id is not None and person_id in phi_data:
        patient_phi = phi_data[person_id]
        file_deny_lists[fp] = build_deny_list_from_phi(patient_phi, header_deny)
        dob = patient_phi.get("dob")
        file_dob_patterns[fp] = _build_dob_patterns(dob) if dob else []
    else:
        file_deny_lists[fp] = header_deny
        file_dob_patterns[fp] = []

print("Phase 1 complete: all deny lists built.")

### Cell 5: Build Work DataFrame

In [0]:
# Create work items for Spark distribution
work_rows = []
for fp in valid_files:
    person_id = linking_map.get(fp)
    accession = file_headers[fp].get("accession") or ""
    mode = "selective" if person_id is not None else "blanket"
    work_rows.append({
        "file_path": fp,
        "deny_list_json": json.dumps(file_deny_lists.get(fp, [])),
        "dob_patterns_json": json.dumps(file_dob_patterns.get(fp, [])),
        "person_id": str(person_id) if person_id else None,
        "mode": mode,
        "accession_nbr": accession,
    })

work_schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("deny_list_json", StringType(), False),
    StructField("dob_patterns_json", StringType(), False),
    StructField("person_id", StringType(), True),
    StructField("mode", StringType(), False),
    StructField("accession_nbr", StringType(), True),
])

work_df = spark.createDataFrame(work_rows, schema=work_schema)

# Repartition to match cluster parallelism
num_gpus = max(1, sc.defaultParallelism)
work_df = work_df.repartition(num_gpus)

print(f"Work DataFrame: {len(work_rows)} files across {num_gpus} partitions")
work_df.show(5, truncate=60)

### Cell 6: Distributed OCR Worker Function (`mapInPandas`)

In [0]:
# Output schema for OCR log rows returned by workers
_OCR_LOG_SCHEMA = StructType([
    StructField("source_file", StringType(), True),
    StructField("accession_nbr", StringType(), True),
    StructField("person_id", StringType(), True),
    StructField("line_index", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("bounding_box", StringType(), True),
    StructField("has_pii", BooleanType(), True),
    StructField("action", StringType(), True),
    StructField("pii_details", StringType(), True),
    StructField("filter_reason", StringType(), True),
    StructField("processed_at", StringType(), True),
    StructField("output_path", StringType(), True),
    StructField("error", StringType(), True),
])

# Broadcast config values so workers don't need widget access
_bc_preprocessing = sc.broadcast(PREPROCESSING)
_bc_min_text_length = sc.broadcast(MIN_TEXT_LENGTH)
_bc_padding = sc.broadcast(PADDING)
_bc_output_folder = sc.broadcast(OUTPUT_FOLDER)
_bc_input_folder = sc.broadcast(INPUT_FOLDER)


def process_batch(batch_iter: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Worker function: loads OCR engine once, processes all assigned images.

    Reads DICOM -> OCR -> PII detect -> Redact -> Save -> yields log rows.
    """
    import os
    import re
    import math
    import json
    import datetime
    import numpy as np
    import cv2
    import SimpleITK as sitk
    from pathlib import Path

    os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.8"

    from paddleocr import PaddleOCR

    # Load OCR engine ONCE per task (Iterator pattern)
    # PaddleOCR 2.8.x API: use_gpu=True, lang="en"
    ocr_engine = PaddleOCR(
        lang="en",
        use_gpu=True,
        use_angle_cls=False,
        det_limit_side_len=960,
        show_log=False,
    )

    preprocessing = _bc_preprocessing.value
    min_text_length = _bc_min_text_length.value
    padding = _bc_padding.value
    output_folder = _bc_output_folder.value
    input_folder = _bc_input_folder.value

    processed_at = datetime.datetime.now().isoformat(timespec="seconds")

    def _ensure_dir(path):
        """Create directory on Volumes — ignore errors since FUSE may auto-create on write."""
        try:
            os.makedirs(path, exist_ok=True)
        except OSError:
            pass

    for batch_df in batch_iter:
        results = []

        for _, row in batch_df.iterrows():
            file_path = row["file_path"]
            deny_list = json.loads(row["deny_list_json"])
            dob_patterns = json.loads(row["dob_patterns_json"])
            person_id = row["person_id"]
            mode = row["mode"]
            accession_nbr = row["accession_nbr"] or ""
            rel_path = os.path.relpath(file_path, input_folder)

            try:
                # Read DICOM
                image = sitk.ReadImage(file_path)
                image_array = sitk.GetArrayFromImage(image)
                target_slice = image_array[0] if image_array.ndim == 3 else image_array

                if mode == "selective":
                    # ── Selective mode: OCR + PII detection ──
                    ocr_results = run_ocr_structured(ocr_engine, target_slice, preprocessing=preprocessing)
                    filtered = filter_ocr_results(ocr_results, min_length=min_text_length)

                    candidates = {k: v for k, v in filtered.items() if not v.get("filtered")}
                    pii_detected = detect_pii_in_ocr(candidates, deny_list)

                    # Check DOB patterns
                    if dob_patterns:
                        for line_id, data in pii_detected.items():
                            if not data["has_pii"]:
                                for pat in dob_patterns:
                                    if re.search(pat, data["text"], re.IGNORECASE):
                                        data["has_pii"] = True
                                        data["pii_details"].append({
                                            "entity_text": data["text"],
                                            "type": "DOB_PATTERN",
                                            "source": "dob_regex",
                                        })
                                        break

                    pii_report = {**filtered}
                    pii_report.update(pii_detected)

                else:
                    # ── Blanket mode: detect all text regions, redact everything ──
                    ocr_results = run_detection_only(ocr_engine, target_slice, preprocessing=preprocessing)
                    pii_report = filter_ocr_results(ocr_results, min_length=0)
                    # All detected regions already have has_pii=True from run_detection_only

                # Redact pixel data
                redacted_slice = redact_pii_from_image(target_slice, pii_report, padding=padding)

                # Build output DICOM
                if image_array.ndim == 3 and redacted_slice.ndim == 2:
                    redacted_for_sitk = redacted_slice[np.newaxis, ...]
                else:
                    redacted_for_sitk = redacted_slice
                redacted_sitk = sitk.GetImageFromArray(redacted_for_sitk)
                redacted_sitk.CopyInformation(image)

                # Copy all metadata then strip PII tags
                for key in image.GetMetaDataKeys():
                    redacted_sitk.SetMetaData(key, image.GetMetaData(key))
                strip_dicom_pii(redacted_sitk)

                # Save output (preserve subdirectory structure)
                rel_dir = os.path.dirname(rel_path)
                base_name = Path(rel_path).stem
                out_subdir = os.path.join(output_folder, rel_dir)
                _ensure_dir(out_subdir)
                dcm_out = os.path.join(out_subdir, f"{base_name}_anon.dcm")
                sitk.WriteImage(redacted_sitk, dcm_out)

                # Emit log rows (one per OCR region)
                for line_id, data in pii_report.items():
                    if data.get("filtered"):
                        action = f"skipped ({data.get('filter_reason', '')})"
                    elif data["has_pii"]:
                        action = "redacted"
                    else:
                        action = "kept"

                    results.append({
                        "source_file": rel_path,
                        "accession_nbr": accession_nbr,
                        "person_id": person_id,
                        "line_index": int(line_id.split("_")[1]) if "_" in line_id else 0,
                        "text": data.get("text", ""),
                        "bounding_box": json.dumps(data.get("bounding_box", {})),
                        "has_pii": data.get("has_pii", False),
                        "action": action,
                        "pii_details": json.dumps(data.get("pii_details", [])),
                        "filter_reason": data.get("filter_reason") or "",
                        "processed_at": processed_at,
                        "output_path": dcm_out,
                        "error": None,
                    })

                # If no OCR regions at all, still emit one row so the file is tracked
                if not pii_report:
                    results.append({
                        "source_file": rel_path,
                        "accession_nbr": accession_nbr,
                        "person_id": person_id,
                        "line_index": 0,
                        "text": "",
                        "bounding_box": "{}",
                        "has_pii": False,
                        "action": "no_text_detected",
                        "pii_details": "[]",
                        "filter_reason": "",
                        "processed_at": processed_at,
                        "output_path": dcm_out,
                        "error": None,
                    })

            except Exception as e:
                results.append({
                    "source_file": rel_path,
                    "accession_nbr": accession_nbr,
                    "person_id": person_id,
                    "line_index": 0,
                    "text": "",
                    "bounding_box": "{}",
                    "has_pii": False,
                    "action": "error",
                    "pii_details": "[]",
                    "filter_reason": "",
                    "processed_at": processed_at,
                    "output_path": None,
                    "error": str(e),
                })

        if results:
            yield pd.DataFrame(results)
        else:
            yield pd.DataFrame(columns=[
                "source_file", "accession_nbr", "person_id", "line_index",
                "text", "bounding_box", "has_pii", "action",
                "pii_details", "filter_reason", "processed_at",
                "output_path", "error",
            ])

    # No ocr.close() in 2.8.x — engine cleaned up on GC
    del ocr_engine

### Cell 7: Execute & Collect Results

In [0]:
# Ensure output folder exists via dbutils (UC Volumes don't support os.makedirs)
dbutils.fs.mkdirs(OUTPUT_FOLDER)

# Execute distributed processing
print(f"Starting distributed OCR across {num_gpus} GPU tasks...")
result_df = work_df.mapInPandas(process_batch, schema=_OCR_LOG_SCHEMA)

# Materialize results — this triggers the distributed work
result_pdf = result_df.toPandas()
print(f"Processing complete: {len(result_pdf)} log rows collected")

In [0]:
# ── Save OCR log to Delta ─────────────────────────────────────────────────────

log_columns = [
    "source_file", "accession_nbr", "person_id", "line_index",
    "text", "bounding_box", "has_pii", "action",
    "pii_details", "filter_reason", "processed_at",
]

if not result_pdf.empty:
    catalog, schema_name, _ = OCR_LOG_TABLE.split(".")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

    log_schema = StructType([
        StructField("source_file", StringType(), True),
        StructField("accession_nbr", StringType(), True),
        StructField("person_id", StringType(), True),
        StructField("line_index", IntegerType(), True),
        StructField("text", StringType(), True),
        StructField("bounding_box", StringType(), True),
        StructField("has_pii", BooleanType(), True),
        StructField("action", StringType(), True),
        StructField("pii_details", StringType(), True),
        StructField("filter_reason", StringType(), True),
        StructField("processed_at", StringType(), True),
    ])

    log_df = spark.createDataFrame(result_pdf[log_columns], schema=log_schema)
    log_df.write.mode("append").option("mergeSchema", "true").saveAsTable(OCR_LOG_TABLE)
    print(f"Saved {len(result_pdf)} rows to {OCR_LOG_TABLE}")
else:
    print("No OCR log rows to save.")

In [0]:
# ── Summary ───────────────────────────────────────────────────────────────────

total_files = len(valid_files)
error_count = int(result_pdf["error"].notna().sum()) if not result_pdf.empty else 0
unique_files_processed = result_pdf["source_file"].nunique() if not result_pdf.empty else 0
text_regions = len(result_pdf[result_pdf["action"] != "error"]) if not result_pdf.empty else 0
redacted_regions = int((result_pdf["action"] == "redacted").sum()) if not result_pdf.empty else 0

print("\n" + "=" * 60)
print("ANONYMIZATION SUMMARY (GPU-distributed)")
print("=" * 60)
print(f"  Input folder:      {INPUT_FOLDER}")
print(f"  Output folder:     {OUTPUT_FOLDER}")
print(f"  Total files found: {len(dicom_files)}")
print(f"  Readable:          {total_files}")
print(f"  Skipped:           {skipped}")
print(f"  Linked (selective):{linked_count}")
print(f"  Unlinked (blanket):{unlinked_count}")
print(f"  Files processed:   {unique_files_processed}")
print(f"  Errors:            {error_count}")
print(f"  Text regions:      {text_regions}")
print(f"  Redacted regions:  {redacted_regions}")
print(f"  OCR log table:     {OCR_LOG_TABLE}")
print(f"  GPU partitions:    {num_gpus}")
print("=" * 60)